## Imports

In [1]:
import landseg.adapters.api as api

## Configure Data Ingestion and Run

In [2]:
# NOTE
# This is a demo script to how to configure the data ingestion pipeline
# The purpose is to process raw rasters into stable artifacts (.npz files) mapped to the grid
# The data artifacts are cataloged for access during data preparation; training will not read them
# Typically this is run once unless input data is changed or you want to adjust the settings
# Adjust the parameters and file paths as needed for your specific use case
# Relative file paths are used here for demonstration (data shipped with the codebase)
# Use absolute file paths to ensure consistency across different environments

# init configurator
configurator = api.DataIngestionConfigurator(
    # the root directory for this experiment, where all outputs will be saved
    experiment_root='../experiment/',
    # name of the input dataset, used for naming outputs and logging
    dataset_name='demo_data'
)

# set grid parameters for tiling the input data
configurator.set_grid(
    # EPSG code for the grid reference raster CRS
    crs='EPSG:2958',
    # path to the reference raster used for defining the tiling grid and extent
    reference_raster_fpath='../experiment/input/extent_reference/extent_reference.tif',
    # size of the tiles to be generated in pixels - here we supoort only square tiles
    tile_size=256,
    # used for data augmentation and to avoid edge effects during training
    tile_overlap=128
)

# set domain knowledge layers to be used as additional input channels for the model
configurator.set_domains([
    # each entry should contain the file path and the raster index base
    # ('../experiment/input/domain_knowledge/....tif', 1),
    # ('../experiment/input/domain_knowledge/....tif', 1)
])

# set training data for model development (i.e., training and validation)
configurator.set_model_dev_data(
    model_dev_image='../experiment/input/data/demo_data/dev_image.tif',
    model_dev_label='../experiment/input/data/demo_data/dev_label.tif',
    data_config='../experiment/input/data/demo_data/config.json',
)

# set test holdout data for final evaluation of the trained model
configurator.set_test_holdout_data(
    # optional - if not provided, model evaluation pipeline will fail
    test_holdout_image='../experiment/input/data/demo_data/test_image.tif',
    test_holdout_label='../experiment/input/data/demo_data/test_label.tif',
)

# run data ingestion
api.run(configurator.running_root_config)

2026-06-20 21:16:14,213-api-INFO	- Running pipeline: data-ingest
2026-06-20 21:16:48,834-ingest.wgrid-INFO	- World grid saved ../experiment//artifacts/demo_data/foundation\world_grids\grid_row_256_128_col_256_128.json
2026-06-20 21:16:49,058-ingest.dblks-INFO	- Try to load mapped windows from grid_row_256_128_col_256_128
2026-06-20 21:16:51,782-ingest.dblks-INFO	- Mapped windows from grid_row_256_128_col_256_128 created
2026-06-20 21:16:51,784-ingest.dblks-INFO	- Rasters mapped to input world grid
2026-06-20 21:16:51,875-ingest.dblks-INFO	- Checking block .npz files
100%|████████████████████████████████████████████████████████████| 400/400 [00:00<00:00, 646.67it/s]
2026-06-20 21:16:53,481-ingest.dblks-INFO	- Found 400 invalid blocks
2026-06-20 21:16:53,482-ingest.dblks-INFO	- Removed 0 damaged files
2026-06-20 21:16:53,483-ingest.dblks-INFO	- 400 blocks to be created
100%|█████████████████████████████████████████████████████████████| 400/400 [05:21<00:00,  1.25it/s]
2026-06-20 21:22:15

## Configure Data Preparation and Run

In [3]:
# NOTE
# Data preparation is the intermediate step between data ingestion and model training
# Main processes include:
# 1. partitioning the model development data into training and validation sets
# 2. performing data augmentation if configured
# 3. normalize data and generate training-scoped artifacts (.npz files) and data schema artifacts
# Once this step is run, the training pipeline can be run directly from the artifacts
# Re-run if you want to adjust how the training view the data at runtime

# init configurator
configurator = api.DataPreparationConfigurator(
    # the root directory for this experiment, where all outputs will be saved
    experiment_root='../experiment/',
    # name of the input dataset, used for naming outputs and logging
    dataset_name='demo_data'
)

# set partitioning strategy for model development data
# partitioning will be done on the tiles from non-overlapping grid blocks
configurator.set_partition(
    # percentage of dev tiles to be used for validation, rest will be used for training
    validation_blocks_ratio=0.15,
    # percentage of dev tiles to be held out for testing
    # if test holdout data is provided, this parameter will be ignored and the holdout data will be used for testing
    test_holdout_blocks_ratio=0.1
)

# set data augmentation and sampling strategy for oversampling under-represented classes in the training data
# augemntation will source from overlapping tiles and will be skipped if tile_overlap is set to 0 in the grid configuration
# set target_head to None or leave reward_classes dictionary empty for no oversampling
configurator.set_oversampling(
    target_head=None,
    # class ID: score value for oversampling (higher value means more oversampling)
    # make sure the class ID matches the label values in the training data
    reward_classes={}
)

# run data preparation
api.run(configurator.running_root_config)

2026-06-20 21:26:04,780-api-INFO	- Running pipeline: data-prepare
2026-06-20 21:26:04,846-prep.split-INFO	- Split data blocks


Class distributions:
              cls_1  cls_2  cls_3  cls_4  cls_5  cls_6  cls_7  cls_8  cls_9  cls_10
Global:       0.872, 0.016, 0.023, 0.069, 0.010, 0.000, 0.009, 0.000, 0.000, 0.000
Split_train:  0.873, 0.016, 0.023, 0.069, 0.010, 0.000, 0.009, 0.000, 0.000, 0.000
Split_val:    0.865, 0.017, 0.027, 0.070, 0.012, 0.000, 0.009, 0.000, 0.000, 0.000
Split_test:   0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000


100%|████████████████████████████████████████████████████████████| 185/185 [00:00<00:00, 577.08it/s]
2026-06-20 21:26:06,072-prep.split-INFO	- ----------Blocks Count: 
2026-06-20 21:26:06,073-prep.split-INFO	-        Training blocks: 54
2026-06-20 21:26:06,073-prep.split-INFO	-      Validation blocks: 10
2026-06-20 21:26:06,074-prep.split-INFO	-            Test blocks: 2
2026-06-20 21:26:06,074-prep.split-INFO	- -----Hydration Results: 
2026-06-20 21:26:06,075-prep.split-INFO	- Prev. base label count:  3,656,812     68,679     97,405    291,457     42,499          0     36,285        803        364          0
2026-06-20 21:26:06,075-prep.split-INFO	- Curr. base label count:  3,089,674     57,551     79,846    245,527     34,858          0     30,488        677        323          0
2026-06-20 21:26:06,076-prep.split-INFO	-             - increase:   -567,138    -11,128    -17,559    -45,930     -7,641          0     -5,797       -126        -41          0
2026-06-20 21:26:06,077-prep.sp